In [ ]:
# 核心库
!pip install numpy pandas matplotlib tqdm pillow --quiet


In [ ]:
# 进度条和交互组件
!pip install ipywidgets --quiet
# 如果你使用 JupyterLab，也请启用 widget 扩展
#!jupyter nbextension enable --py widgetsnbextension


In [ ]:
# PyTorch（CPU 版本）
!pip install torch torchvision --quiet


In [ ]:
import os
import uuid
import shutil
import json
import copy
from datetime import datetime
import zipfile
import io
import requests
import random


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from matplotlib.pyplot import imshow
from tqdm import tqdm
from ipywidgets import IntProgress
import time 


In [ ]:
import torch
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader,random_split
from torch.optim import lr_scheduler
from torchvision import transforms
import torch.nn as nn
torch.manual_seed(0)
from torchvision.datasets import ImageFolder


In [ ]:
def plot_stuff(COST, ACC):
    """
    使用两个 y 轴在同一图中绘制训练代价（损失）和验证准确率。
    
    参数：
    COST（列表或数组）：每次迭代（或轮次）的总训练损失
    ACC（列表或数组）：每次迭代（或轮次）的验证准确率
    """
    
    # 创建新图和主轴 (ax1)
    fig, ax1 = plt.subplots()
    
    # 在左侧主轴 y 轴上绘制训练损失
    color = 'tab:red'
    ax1.plot(COST, color=color)
    ax1.set_xlabel('迭代', color=color)            # x 轴标签
    ax1.set_ylabel('总损失', color=color)           # 左侧 y 轴标签
    ax1.tick_params(axis='y', labelcolor=color)         # 设置 y 轴刻度颜色

    # 创建共享同一 x 轴的次 y 轴 (ax2)
    ax2 = ax1.twinx()
    
    # 在右侧次 y 轴上绘制验证准确率
    color = 'tab:blue'
    ax2.set_ylabel('准确率', color=color)             # 右侧 y 轴标签
    ax2.plot(ACC, color=color)
    ax2.tick_params(axis='y', labelcolor=color)

    # 调整布局以防止 y 轴标签被裁剪
    fig.tight_layout()
    
    # 显示组合图
    plt.show()


In [ ]:
def imshow_(inp, title=None):
    """
    显示撤销归一化后的张量图像。
    
    参数：
    - inp（Tensor）：形状为 [C, H, W] 的图像张量，通常已归一化。
    - title（字符串，可选）：图像显示的标题。
    """
    # 从 [C, H, W] 转换为 [H, W, C] 并转为 NumPy 数组
    inp = inp.permute(1, 2, 0).numpy()
    print("图像形状:", inp.shape)

    # 撤销归一化（ImageNet 均值和标准差）
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean

    # 将值裁剪到 [0, 1] 范围以便显示
    inp = np.clip(inp, 0, 1)

    # 显示图像
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # 短暂暂停以更新 GUI
    plt.show()


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("设备类型为", device)


In [ ]:
# ── 路径配置（按你的实际情况修改）──
zip_path        = "Pistachio_Image_Dataset.zip"          # 当前目录下的压缩包
extract_dir     = "Pistachio_Image_Dataset"              # 解压后的根文件夹名

# ── 1. 自动解压 ──
zip_path = "Pistachio_Image_Dataset.zip"

if not os.path.exists("Pistachio_Image_Dataset"):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(".")
    print("解压完成。")
else:
    print("文件夹已存在，跳过解压。")


In [ ]:

# ── 2. 自动识别图像源目录 ──
extract_dir = "Pistachio_Image_Dataset"

if os.path.exists(os.path.join(extract_dir, "Kirmizi_Pistachio")):
    source_dir = extract_dir
else:
    sub = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))][0]
    source_dir = os.path.join(extract_dir, sub)

print(f"图像源目录: {source_dir}")


In [ ]:

# ── 3. 收集图像 ──
label_to_images = {}
for label in ["Kirmizi_Pistachio", "Siirt_Pistachio"]:
    label_path = os.path.join(source_dir, label)
    if os.path.isdir(label_path):
        exts = (".jpg", ".jpeg", ".png", ".bmp")
        images = [f for f in os.listdir(label_path) if f.lower().endswith(exts)]
        label_to_images[label] = images
        print(f"  {label}: {len(images)} 张")

In [ ]:
# ── 4. 划分并复制 ──
train_ratio = 0.8
random.seed(42)
output_dir = "pistachio_dataset"

for label, image_list in label_to_images.items():
    random.shuffle(image_list)
    cutoff = int(len(image_list) * train_ratio)
    train_images = image_list[:cutoff]
    val_images = image_list[cutoff:]

    for split, split_images in zip(["train", "val"], [train_images, val_images]):
        out_path = os.path.join(output_dir, split, label)
        os.makedirs(out_path, exist_ok=True)
        for img_name in split_images:
            src = os.path.join(source_dir, label, img_name)
            dst = os.path.join(out_path, img_name)
            shutil.copy2(src, dst)


In [ ]:

# ── 5. 输出统计 ──
print("\n划分完成！")
for label in label_to_images:
    train_n = len(os.listdir(os.path.join(output_dir, "train", label)))
    val_n = len(os.listdir(os.path.join(output_dir, "val", label)))
    print(f"  {label}: train={train_n}  val={val_n}  总计={train_n+val_n}")


In [ ]:
# 定义要应用于每张图像的一系列变换
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 将所有图像调整为 224×224 像素（预训练模型的标准输入大小）
    transforms.ToTensor(),          # 将 PIL 图像转换为形状为 [C, H, W]、取值在 [0, 1] 的 PyTorch 张量
    transforms.Normalize(           # 使用 ImageNet 的均值和标准差对每个通道（RGB）进行归一化
        [0.485, 0.456, 0.406],      # 红、绿、蓝通道的均值
        [0.229, 0.224, 0.225]       # 红、绿、蓝通道的标准差
    )
])

# 从文件夹结构加载训练数据集并应用定义的变换
train_dataset = ImageFolder("pistachio_dataset/train", transform=transform)

# 从文件夹结构加载验证数据集并应用相同的变换
#（注意：这里不应用数据增强——只进行大小调整、张量转换和归一化）
val_dataset = ImageFolder("pistachio_dataset/val", transform=transform)


In [ ]:
i = 0
for x, y in val_dataset:                     # 遍历验证数据集
    imshow_(x, f"y = {y}")               # 显示图像及其标签
    i += 1                               # 计数器递增
    if i == 3:                           # 显示 3 张图像后停止
        break


## 超参数

In [ ]:
n_epochs=10
batch_size=32
lr=0.000001
momentum=0.9
lr_scheduler=True


## 加载模型并训练

In [ ]:
def train_model(model, train_loader, validation_loader, criterion, optimizer, n_epochs, print_=True):
    loss_list = []        # 存储每个轮次的平均训练损失
    accuracy_list = []    # 存储每个轮次的验证准确率
    correct = 0

    n_test = len(val_dataset)  # 验证样本总数
    accuracy_best = 0      # 跟踪最佳验证准确率
    best_model_wts = copy.deepcopy(model.state_dict())  # 备份最佳模型权重

    print("第一个轮次可能需要几分钟")

    for epoch in tqdm(range(n_epochs)):  # 遍历每个轮次
        loss_sublist = []  # 存储该轮次的每个批次损失

        # 训练阶段
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            model.train()  # 将模型设置为训练模式

            z = model(x)   # 前向传播
            loss = criterion(z, y)  # 计算损失
            loss_sublist.append(loss.item())

            loss.backward()       # 反向传播
            optimizer.step()      # 更新权重
            optimizer.zero_grad() # 重置梯度

        print(f"轮次 {epoch + 1} 完成")

        # 如果定义了学习率调度器，则调整学习率
        scheduler.step()

        # 存储该轮次的平均训练损失
        loss_list.append(np.mean(loss_sublist))

        # 验证阶段
        correct = 0
        model.eval()  # 将模型设置为评估模式
        with torch.no_grad():
            for x_test, y_test in validation_loader:
                x_test, y_test = x_test.to(device), y_test.to(device)
                z = model(x_test)
                _, yhat = torch.max(z.data, 1)
                correct += (yhat == y_test).sum().item()

        accuracy = correct / n_test
        accuracy_list.append(accuracy)

        # 保存最佳模型
        if accuracy > accuracy_best:
            accuracy_best = accuracy
            best_model_wts = copy.deepcopy(model.state_dict())

        # 输出训练进度
        if print_:
            print("学习率:", optimizer.param_groups[0]['lr'])
            print(f"验证损失（轮次 {epoch + 1}）: {np.mean(loss_sublist):.4f}")
            print(f"验证准确率（轮次 {epoch + 1}）: {accuracy:.4f}")

    # 返回前加载最佳模型权重
    model.load_state_dict(best_model_wts)
    return accuracy_list, loss_list, model


In [ ]:
model = models.resnet18(pretrained=True)


- 含义：遍历模型的所有参数（权重和偏置），将 requires_grad（是否需要计算梯度）设为 False。
- 作用：在后续训练中，这些层的参数不会被更新（即“冻结”）。这能保留预训练学到的通用特征，防止在小数据集（你的开心果数据）上训练时破坏原有权重（过拟合），同时大幅减少训练时间和显存消耗。

In [ ]:
for param in model.parameters():
        param.requires_grad = False
    

In [ ]:
n_classes = len(train_dataset.classes)
print(n_classes)


- 由于上一步冻结了旧参数，这个新替换的 fc 层默认是未冻结的（requires_grad=True），它将成为训练阶段唯一被更新的部分

In [ ]:
model.fc = nn.Linear(512, n_classes)


In [ ]:
model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()


In [ ]:
train_loader = torch.utils.data.DataLoader(dataset=train_dataset , batch_size=batch_size,shuffle=True)
validation_loader= torch.utils.data.DataLoader(dataset=val_dataset , batch_size=1)


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)


In [ ]:
if lr_scheduler:
    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        base_lr=0.001,        # 最小学习率
        max_lr=0.01,          # 最大学习率
        step_size_up=5,       # 学习率从 base_lr 增加到 max_lr 所需的步数
        mode="triangular2"    # 学习率按三角周期变化，每个周期后将 max_lr 减半
    )


In [ ]:
# 开始计时
start_datetime = datetime.now()
start_time = time.time()

# 训练模型
accuracy_list, loss_list, model = train_model(
    model, train_loader, validation_loader, criterion, optimizer, n_epochs=n_epochs
)

# 结束计时
end_datetime = datetime.now()
elapsed_time = time.time() - start_time

# 输出结果
print("训练完成。")
print(f"开始时间     : {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"结束时间       : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"已用时间   : {elapsed_time:.2f} 秒")


In [ ]:
# 将模型保存到 model.pt
torch.save(model.state_dict(), 'model.pt')


In [ ]:
plot_stuff(loss_list,accuracy_list)


In [ ]:
# 定义类别名称（与训练时一致）
class_names = ['Kirmizi_Pistachio', 'Siirt_Pistachio']
# 创建与训练时相同的模型架构
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, 2)  # 2 个类别：hotdog / nothotdog

# 加载训练好的权重
model.load_state_dict(torch.load("model.pt", map_location=torch.device('cpu')))
model.eval()  # 设置为评估模式


In [ ]:
# 定义图像变换（必须与训练时一致）
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整图像大小
    transforms.ToTensor(),  # 转换为张量
    transforms.Normalize([0.485, 0.456, 0.406],  # 归一化（与 ImageNet/预训练模型相同）
                         [0.229, 0.224, 0.225])
])


In [ ]:
image_path = "kirmizi (12).jpg"  # 替换为你的图像路径

# 打开并转换为 RGB
image = Image.open(image_path).convert("RGB")

# 应用变换
input_tensor = transform(image).unsqueeze(0)  # 添加批次维度


In [ ]:
with torch.no_grad():
    outputs = model(input_tensor)
    predicted_class = torch.argmax(outputs, 1).item()
# 显示结果
print(f"The image was classified as: {class_names[predicted_class]}")
# 显示带有预测标签的图像
plt.imshow(image)  # 原始 PIL 图像
plt.title(f"Predicted: {class_names[predicted_class]}")
plt.axis("off")
plt.show()
